In [192]:
import pandas as pd
from pathlib import Path

In [193]:
# load raw hospital files
RAW_DIR = Path("data/raw/hospitals")

files = {
    2021: RAW_DIR / "Hospital_General_Information_2021.csv",
    2022: RAW_DIR / "Hospital_General_Information_2022.csv",
    2023: RAW_DIR / "Hospital_General_Information_2023.csv",
}

dfs = []

for year, path in files.items():
    df = pd.read_csv(path)
    df["year"] = year
    dfs.append(df)

hospitals_raw = pd.concat(dfs, ignore_index=True)

In [194]:
[c for c in hospitals_raw.columns if "emerg" in c.lower() or "ehr" in c.lower() or "meaningful" in c.lower() or "electronic" in c.lower()]

['Emergency Services', 'Meets criteria for promoting interoperability of EHRs']

In [196]:
# sanity check
hospitals_raw.shape
hospitals_raw["year"].value_counts().sort_index()

year
2021    5325
2022    5307
2023    5446
Name: count, dtype: int64

In [197]:
hospitals_raw["Emergency Services"].value_counts(dropna=False).head(20)

Emergency Services
Yes    13472
No      2606
Name: count, dtype: int64

In [198]:
hospitals_raw["Meets criteria for promoting interoperability of EHRs"].value_counts(dropna=False).head(20)

Meets criteria for promoting interoperability of EHRs
Y      11663
NaN     4415
Name: count, dtype: int64

In [199]:
# standardize column names
hospitals_raw.columns = (
    hospitals_raw.columns
        .str.strip()
        .str.lower()
        .str.replace(r"[^\w]+", "_", regex=True)
)

In [201]:
# select essential columns only
cols_keep = [
    "facility_id",
    "facility_name",
    "state",
    "city",
    "zip_code",
    "year",
    "hospital_type",
    "hospital_ownership",
    "emergency_services",
    "meets_criteria_for_promoting_interoperability_of_ehrs",
    "hospital_overall_rating"
]

hospitals = hospitals_raw[cols_keep].copy()

In [202]:
# rename columns clearly
hospitals = hospitals.rename(columns={
    "facility_id": "hospital_id",
    "facility_name": "hospital_name",
    "city": "hospital_city",
    "zip_code": "hospital_zip",
    "emergency_services": "hospital_has_emergency_services",
    "meets_criteria_for_promoting_interoperability_of_ehrs": "hospital_has_ehr_interoperability",
    "hospital_overall_rating": "hospital_overall_rating"
})

In [204]:
# normalize binary indicators

binary_true = {"Yes", "Y"}
binary_false = {"No", "N"}

def normalize_binary(x):
    if pd.isna(x):
        return None
    x = str(x).strip()
    if x in binary_true:
        return 1
    if x in binary_false:
        return 0
    return None

hospitals["hospital_has_emergency_services"] = (
    hospitals["hospital_has_emergency_services"]
    .apply(normalize_binary)
)

hospitals["hospital_has_ehr_interoperability"] = (
    hospitals["hospital_has_ehr_interoperability"]
    .apply(normalize_binary)
)

In [205]:
# ZIP codes should be treated as strings (preserve leading zeros)
hospitals["hospital_zip"] = (
    hospitals["hospital_zip"]
    .astype(str)
    .str.zfill(5)
)

In [206]:
# convert rating to numeric; non-numeric entries become NaN
hospitals["hospital_overall_rating"] = pd.to_numeric(
    hospitals["hospital_overall_rating"],
    errors="coerce"
)

In [207]:
# coverage check: fraction of hospitals with non-missing binary indicators
hospitals[
    ["hospital_has_emergency_services", "hospital_has_ehr_interoperability"]
].notna().mean()

hospital_has_emergency_services      1.000000
hospital_has_ehr_interoperability    0.725401
dtype: float64

## Interpretation
- Emergency services: 100% coverage after normalization
→ Every hospital reports Yes/No (good).
- EHR interoperability: ~72.5% coverage
→ ~27.5% of hospitals do not report this field (expected and realistic).

In [209]:
# confirm normalization results (should be 1/0 with NaN where not reported)
hospitals["hospital_has_emergency_services"].value_counts(dropna=False)
hospitals["hospital_has_ehr_interoperability"].value_counts(dropna=False)

hospital_has_ehr_interoperability
1.0    11663
NaN     4415
Name: count, dtype: int64

In [210]:
# proportion of missing hospital overall ratings (expected—many hospitals lack ratings)
hospitals["hospital_overall_rating"].isna().mean()

np.float64(0.41056101505162335)

In [211]:
# integrity checks

# one row per hospital-year
hospitals.duplicated(subset=["hospital_id", "year"]).sum()

# number of states
hospitals["state"].nunique()

# ownership distribution
hospitals["hospital_ownership"].value_counts()


hospital_ownership
Voluntary non-profit - Private                 6772
Proprietary                                    3164
Government - Hospital District or Authority    1574
Government - Local                             1273
Voluntary non-profit - Other                   1142
Voluntary non-profit - Church                   898
Government - State                              617
Physician                                       232
Veterans Health Administration                  137
Government - Federal                            133
Department of Defense                           105
Tribal                                           31
Name: count, dtype: int64

In [213]:
CLEAN_DIR = Path("data/clean/hospitals")
CLEAN_DIR.mkdir(parents=True, exist_ok=True)

hospitals.to_csv(
    CLEAN_DIR / "hospitals_clean_2021_2023.csv",
    index=False
)